# 03. Normalización básica de textos con spaCy y REGEX

En este notebook vamos a ver un primer ejemplo de **normalización de textos** utilizando **spaCy** y **expresiones regulares**.

El objetivo es transformar una frases escritsa en lenguaje natural en una lista de tokens más limpia y estructurada. Para ello aplicaremos varias operaciones básicas:

* Detectar un número de pasaporte mediante una expresión regular.
* Sustituir el número de pasaporte por un marcador temporal.
* Procesar el texto con spaCy.
* Unir entidades nombradas formadas por varias palabras, como `Nueva York`, `Real Madrid`, etc. en un único token.
* Eliminar signos de puntuación, espacios y stopwords.
* Obtener el lema de cada token.
* Pasar los tokens a minúsculas.
* Sustituir espacios internos por guion bajo en entidades compuestas.

---

## 1. Cargamos el modelo

In [1]:
import re
import spacy

nlp = spacy.load("es_core_news_sm")

## 2. Texto de partida y detección del pasaporte

Partimos de la siguiente frase que contiene una persona, un lugar formado por varias palabras y un número de pasaporte:

```text
María viajó a Nueva York con el pasaporte ABC123456.
```

Vamos a usar una expresión regular para detectar el número de pasaporte. En este ejemplo usaremos un formato simplificado: tres letras mayúsculas seguidas de seis números.

Cuando el patrón coincida, sustituiremos el número de pasaporte por el marcador `PASAPORTEDOC`. Este marcador nos permite ocultar el valor original del pasaporte y conservar la información de que en esa posición había un documento personal.


In [2]:
texto = "María viajó a Nueva York con el pasaporte ABC123456."

PASAPORTE_REGEX = re.compile(r"\b[A-Z]{3}\d{6}\b")
MARCADOR_PASAPORTE = "PASAPORTEDOC"

texto = PASAPORTE_REGEX.sub(f" {MARCADOR_PASAPORTE} ", texto)
texto

'María viajó a Nueva York con el pasaporte  PASAPORTEDOC .'

## 3. Procesamiento con spaCy y unión de entidades

Pasamos a procesar el texto con spaCy. El resultado será un objeto `Doc`, que contiene los tokens y la información lingüística calculada por el modelo.

Después usamos `doc.retokenize()` para unir las entidades nombradas detectadas por spaCy en un único token. Esto es útil cuando una entidad está formada por varias palabras. Por ejemplo, `Nueva York` no nos interesa como dos tokens separados (`Nueva` y `York`), sino como una única entidad.

De esta forma, el texto queda mejor preparado para los siguientes pasos de normalización.


In [3]:
# Procesamos el texto con spaCy
doc = nlp(texto)

# Unimos las entidades nombradas para tratarlas como un único token
with doc.retokenize() as retokenizer:
    for ent in doc.ents:
        retokenizer.merge(ent)

print(f"Texto tokenizado: {[token.text for token in doc]}")

Texto tokenizado: ['María', 'viajó', 'a', 'Nueva York', 'con', 'el', 'pasaporte', ' ', 'PASAPORTEDOC', '.']


## 4. Normalización de tokens

Ahora vamos a construir una lista con los tokens normalizados.

Para ello eliminaremos los tokens que sean:

* Signos de puntuación (`is_punct`).
* Espacios en blanco (`is_space`).
* Stopwords (`is_stop`), como `a`, `con`, `el` o `la`.

Además, solo conservaremos tokens que cumplan al menos una de estas condiciones:

* Que sean alfabéticos (`is_alpha`).
* Que sean una entidad nombrada (`ent_type_` no vacío).

Por último, a cada token conservado le aplicaremos estas transformaciones:

* Obtener su lema (`lemma_`).
* Pasarlo a minúsculas (`lower()`).
* Sustituir espacios internos por guion bajo (`replace(" ", "_")`).

Esto último es importante para entidades compuestas. Por ejemplo, si spaCy ha unido `Nueva York` como una única entidad, la convertiremos en `nueva_york`.


In [4]:
tokens = [
    token.lemma_.lower().replace(" ", "_")
    for token in doc
    if not (token.is_punct or token.is_space or token.is_stop)
    and (token.is_alpha or token.ent_type_)
]

tokens

['maría', 'viajar', 'nueva_york', 'pasaporte', 'pasaportedoc']

## 5. Resultado obtenido

El resultado esperado es una lista de tokens normalizados parecida a esta:

```python
['maría', 'viajar', 'nueva_york', 'pasaporte', 'pasaportedoc']
```

Podemos observar varias transformaciones importantes:

* `viajó` se ha convertido en `viajar`, que es su lema.
* `Nueva York` se ha mantenido como una única entidad y se ha transformado en `nueva_york`.
* Las stopwords como `a`, `con` y `el` se han eliminado.
* El número de pasaporte `ABC123456` se ha sustituido por el marcador `pasaportedoc`.


---

# Normalización de un Corpus

Una vez entendido el proceso paso a paso con una única frase, podemos encapsular la lógica en una función y aplicarla sobre una lista de textos o Corpus.

En este ejemplo vamos a normalizar tres frases. Para cada una de ellas aplicaremos el mismo flujo de trabajo: 
1. detección de números de pasaporte con REGEX
2. procesamiento con spaCy
3. unión de entidades nombradas
4. eliminación de puntuación, espacios y stopwords, conversión a minúsculas y obtención de lemas.

El resultado final será una **lista de listas**, donde cada texto queda representado como una lista de tokens normalizados.


In [5]:
import re
import spacy

# Cargamos el modelo de español
nlp = spacy.load("es_core_news_sm")

# Lista de textos que queremos normalizar
textos = [
    "El Real Madrid jugará la final en Londres.",
    "María viajó a Nueva York con el pasaporte ABC123456.",
    "Ricardo presentó el pasaporte XYZ987654 en el aeropuerto de Madrid."
]

# Expresión regular para detectar números de pasaporte
PASAPORTE_REGEX = re.compile(r"\b[A-Z]{3}\d{6}\b")

# Marcador que sustituirá al número de pasaporte original
MARCADOR_PASAPORTE = "PASAPORTEDOC"


def normalizar_texto(texto):
    # 1. Sustituimos los números de pasaporte por un marcador
    texto = PASAPORTE_REGEX.sub(f" {MARCADOR_PASAPORTE} ", texto)

    # 2. Procesamos el texto con spaCy
    doc = nlp(texto)

    # 3. Unimos las entidades nombradas para tratarlas como un único token
    with doc.retokenize() as retokenizer:
        for ent in doc.ents:
            retokenizer.merge(ent)

    # 4. Normalizamos los tokens
    tokens = [
        token.lemma_.lower().replace(" ", "_")
        for token in doc
        if not (token.is_punct or token.is_space or token.is_stop)
        and (token.is_alpha or token.ent_type_)
    ]

    return tokens


# Aplicamos la función a todos los textos
textos_tokenizados = [normalizar_texto(texto) for texto in textos]

textos_tokenizados

[['real_madrid', 'jugar', 'londres'],
 ['maría', 'viajar', 'nueva_york', 'pasaporte', 'pasaportedoc'],
 ['ricardo', 'presentar', 'pasaporte', 'pasaportedoc', 'aeropuerto_de_madrid']]